# 06 · Media settings

Every item has to fit in the model's **context of 8,192 tokens**. Text costs about one token per
word piece. Media costs:

| Media | Cost | Setting |
|---|---|---|
| Image | ~270 tokens | `image_tokens` (70 / 140 / **280** / 560 / 1120) |
| Video | ~120 tokens per frame, 1 frame per second, at most 32 frames | `video_frame_tokens`, `video_fps`, `video_max_frames` |
| Audio | 25 tokens per second | none |

Settings live in `EngineConfig`, because vectors made with different settings don't mix.

In [1]:
import os
from pathlib import Path

import embeddinggemma_engine as eg

if Path.cwd().name == "notebooks":
    os.chdir("..")  # run from the repo root, so paths stay short
IMAGES = Path("assets/images")
VIDEOS = Path("assets/videos")
AUDIO = Path("assets/audio")


def mistake(fn):
    """Run fn and print the error the engine raises for it."""
    try:
        fn()
    except eg.EngineError as error:
        print(f"{type(error).__name__}: {error}\n")

In [2]:
engine = eg.EmbeddingEngine()

## Checking the cost before embedding
`count_tokens` reports exactly what each item will use, without running the model.

In [3]:
items = [
    eg.Item.image(IMAGES / "wildlife.jpg"),
    eg.Item.video(VIDEOS / "05_juggle.mp4"),
    eg.Item.audio(AUDIO / "musicwallah-space_station_ambient-sfx-567285.mp3"),
]
for name, count in zip(["photo", "10 s video", "22 s audio"], engine.count_tokens(items, task="similarity")):
    print(f"{name:11} {count.total:5} tokens")

photo         280 tokens
10 s video   1222 tokens
22 s audio    564 tokens


## Cheaper images and frames
Fewer tokens per image means less detail, but more images per item and faster embedding.

In [4]:
light = eg.EmbeddingEngine(eg.EngineConfig(image_tokens=70, video_frame_tokens=70))
for name, count in zip(["photo", "10 s video"], light.count_tokens(items[:2], task="similarity")):
    print(f"{name:11} {count.total:5} tokens")

photo          70 tokens
10 s video    682 tokens


## Long videos
At 1 frame per second, a video longer than 32 seconds has more frames than the limit.
The underlying library would quietly drop frames; the engine stops and asks you to choose.

In [5]:
long_video = VIDEOS / "30s_vid.mp4"  # actually 34.8 s
mistake(lambda: engine.embed([eg.Item.video(long_video)], task="similarity"))

FrameLimitError: item 0: assets/videos/30s_vid.mp4 is 34.8 s, which is 34 frames at video_fps=1.0, more than video_max_frames=32. The processor would silently drop 2 frames. Use Video(..., overflow='subsample') to accept uniform subsampling, or change video_fps / video_max_frames.



Three ways to handle it:

In [6]:
# 1. accept 32 frames spread evenly over the clip
print(engine.count_tokens([eg.Item.video(long_video, overflow="subsample")], task="similarity")[0].total)

# 2. allow more frames
print(eg.EmbeddingEngine(eg.EngineConfig(video_max_frames=40)).count_tokens([eg.Item.video(long_video)],
                                                                            task="similarity")[0].total)

# 3. take fewer frames per second
print(eg.EmbeddingEngine(eg.EngineConfig(video_fps=0.5)).count_tokens([eg.Item.video(long_video)],
                                                                      task="similarity")[0].total)

3906


4150


2076


## A video's sound
By default only the frames are used. `include_audio=True` adds the soundtrack too.

In [7]:
coffee = VIDEOS / "04_coffee.mp4"
frames_only, with_sound = engine.count_tokens(
    [eg.Item.video(coffee), eg.Item.video(coffee, include_audio=True)], task="similarity")
print("frames only:", frames_only)
print("with sound: ", with_sound)

frames only: TokenCount(total=1100, image=0, video=1080, audio=0, other=20)
with sound:  TokenCount(total=1341, image=0, video=1080, audio=239, other=22)


## Audio from memory
Files in any common format work as they are. For audio you already have as an array, also give its
sampling rate. The engine converts it to the mono 16 kHz the model needs.

In [8]:
import numpy as np  # noqa: E402

one_second_of_stereo = np.zeros((2, 44_100), dtype=np.float32)  # (channels, samples) at 44.1 kHz
print(engine.count_tokens([eg.Item.audio(one_second_of_stereo, sampling_rate=44_100)], task="similarity")[0])

TokenCount(total=29, image=0, video=0, audio=25, other=4)


## Loading only what you need
The model has separate parts for images/video and for audio. Leave out what you don't use.

| `modalities` | GPU memory |
|---|---|
| `("text",)` | 0.54 GB |
| `("text", "image", "video")` | 0.89 GB |
| `("text", "audio")` | 1.15 GB |
| all four (default) | 1.50 GB |

In [9]:
text_only = eg.EmbeddingEngine(eg.EngineConfig(modalities=("text",)))
print(text_only.embed(["hello"], task="similarity").vectors.shape)

(1, 768)


## Common mistakes

In [10]:
print("Too much in one item:")
mistake(lambda: engine.embed([eg.Item.of(*[eg.Image(IMAGES / "wildlife.jpg")] * 30)], task="similarity"))

print("An audio array without its sampling rate:")
mistake(lambda: eg.Item.audio(one_second_of_stereo))

print("Asking for the sound of a video that has none:")
mistake(lambda: engine.embed([eg.Item.video(VIDEOS / "05_juggle.mp4", include_audio=True)], task="similarity"))

print("Media the engine wasn't loaded for:")
mistake(lambda: text_only.embed([eg.Item.image(IMAGES / "kitchen.jpg")], task="similarity"))

Too much in one item:


ContextOverflowError: item 0 needs 8342 tokens, more than max_tokens=8192 (image 8280, video 0, audio 0, text+special 62). Shorten it, split it, or lower image_tokens / video_frame_tokens / video_fps.

An audio array without its sampling rate:
InputError: Audio arrays need sampling_rate=... (the processor would silently treat any array as 16 kHz, stretching or squashing it).

Asking for the sound of a video that has none:
MediaError: assets/videos/05_juggle.mp4 has no audio stream.

Media the engine wasn't loaded for:
ModalityNotLoadedError: item 0 uses ['image'], but the engine was configured with modalities=('text',).

